# Cross validation of quantile models

In [ ]:
import sys
import os
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

## Load data

In [ ]:
H_gpr = pd.read_csv("../../_temp/v0/cv.GPR_H.csv")
H_gpqr = pd.read_csv("../../_temp/v0/cv.CenterGapMTGPQR_H.csv")

phi_gpr = pd.read_csv("../../_temp/v0/cv.GPR_phi.csv")
phi_gpqr = pd.read_csv("../../_temp/v0/cv.CenterGapMTGPQR_phi.csv")

## Plot

In [ ]:
def sci_label(value: float, precision: int = 2) -> str:
    if value == 0:
        return f"{0:.{precision}f} × 10^0"
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10**exponent)
    return f"{mantissa:.{precision}f} × 10^{exponent}"

## H

In [ ]:
H_gpr_pinball = np.stack(
    [df["test_pinball_loss"] for (_, df) in H_gpr.groupby("fold")]
).T
H_gpqr_pinball = np.stack(
    [df["test_pinball_loss"] for (_, df) in H_gpqr.groupby("fold")]
).T
H_gpqr_mll = np.stack([df["test_mll_loss"] for (_, df) in H_gpqr.groupby("fold")]).T

### CV loss by epoch

In [ ]:
plt.plot(np.arange(1, len(H_gpr_pinball) + 1), H_gpr_pinball.mean(axis=1), label="GPR")
plt.plot(
    np.arange(1, len(H_gpqr_pinball) + 1), H_gpqr_pinball.mean(axis=1), label="GPQR"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Validation loss by epoch")
plt.legend()
plt.show()

### Minimum CV loss

In [ ]:
names = ["GPR", "GPQR"]
cv_losses = [
    H_gpr_pinball.mean(axis=1).min(),
    H_gpqr_pinball.mean(axis=1).min(),
]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots()
bars_cv = ax.bar(x, cv_losses, width, label="Cross Validation")

for bar in list(bars_cv):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        sci_label(height),
        ha="center",
        va="bottom",
        fontsize=8,
    )

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel("Loss")
ax.set_title("Minimum Cross Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()

### Best epoch

In [ ]:
best_epochs = H_gpqr_mll.argmin(axis=0)
median_best_epoch = int(np.median(best_epochs)) + 1

plt.hist(best_epochs)
plt.axvline(median_best_epoch, ls="--", label=f"Median: {median_best_epoch}")

plt.xlabel("Best epoch")
plt.ylabel("Frequency")
plt.title("Best epoch distribution")
plt.legend()
plt.show()

## phi

In [ ]:
phi_gpr_pinball = np.stack(
    [df["test_pinball_loss"] for (_, df) in phi_gpr.groupby("fold")]
).T
phi_gpqr_pinball = np.stack(
    [df["test_pinball_loss"] for (_, df) in phi_gpqr.groupby("fold")]
).T
phi_gpqr_mll = np.stack([df["test_mll_loss"] for (_, df) in phi_gpqr.groupby("fold")]).T

### CV loss by epoch

In [ ]:
plt.plot(
    np.arange(1, len(phi_gpr_pinball) + 1), phi_gpr_pinball.mean(axis=1), label="GPR"
)
plt.plot(
    np.arange(1, len(phi_gpqr_pinball) + 1), phi_gpqr_pinball.mean(axis=1), label="GPQR"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Validation loss by epoch")
plt.legend()
plt.show()

### Minimum CV loss

In [ ]:
names = ["GPR", "GPQR"]
cv_losses = [
    phi_gpr_pinball.mean(axis=1).min(),
    phi_gpqr_pinball.mean(axis=1).min(),
]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots()
bars_cv = ax.bar(x, cv_losses, width, label="Cross Validation")

for bar in list(bars_cv):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        height,
        sci_label(height),
        ha="center",
        va="bottom",
        fontsize=8,
    )

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel("Loss")
ax.set_title("Minimum Cross Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()

### Best epoch

In [ ]:
best_epochs = phi_gpqr_mll.argmin(axis=0)
median_best_epoch = int(np.median(best_epochs)) + 1

plt.hist(best_epochs)
plt.axvline(median_best_epoch, ls="--", label=f"Median: {median_best_epoch}")

plt.xlabel("Best epoch")
plt.ylabel("Frequency")
plt.title("Best epoch distribution")
plt.legend()
plt.show()